# 1-Aug-26

In [ ]:
# Exam

In [ ]:
# Complete Banking System — OOP + Custom Exceptions

# This version covers all the requirements, including abstract class, encapsulation, inheritance, method overriding, custom exceptions, automatic account numbers, daily withdrawal tracking, overdraft, minimum balance, class methods, and try-except-else-finally.

In [ ]:
# 1.Create a SavingsAccount class with:
 # Account number
# Name
# Balance
# Minimum balance ₹1,000
# 2.Create a CurrentAccount class with:
# Account number
# Name
# Balance
# Overdraft limit ₹10,000
# 3.Implement:
# deposit()
# withdraw()
# get_balance()
# 4.Create custom exceptions:
# InvalidAmountError
# InsufficientBalanceError
# AccountNotFoundError
# 5.Use try-except-else-finally.
# 6.Display account details and total number of accounts.

# Exam tip: Keep the program simple. Focus on class, inheritance, abstraction, private variables, custom exceptions, and method overriding.

In [1]:
from abc import ABC, abstractmethod
from datetime import date


# ============================================================
# 1. CUSTOM EXCEPTIONS
# ============================================================

class InsufficientBalanceError(Exception):
    """Raised when withdrawal amount is greater than available balance."""
    pass


class InvalidAmountError(Exception):
    """Raised when amount is zero or negative."""
    pass


class AccountNotFoundError(Exception):
    """Raised when account number does not exist."""
    pass


class DailyLimitExceededError(Exception):
    """Raised when daily withdrawal limit is crossed."""
    pass


# ============================================================
# 2. ABSTRACT CLASS
# ============================================================

class Account(ABC):

    total_accounts = 0
    next_account_no = 1001

    def __init__(self, account_no, name, balance):
        self.__account_no = account_no
        self.__name = name
        self.__balance = balance

        Account.total_accounts += 1

    # --------------------------------------------------------
    # Abstract methods
    # --------------------------------------------------------

    @abstractmethod
    def deposit(self, amount):
        pass

    @abstractmethod
    def withdraw(self, amount):
        pass

    @abstractmethod
    def get_balance(self):
        pass

    # --------------------------------------------------------
    # Getters
    # --------------------------------------------------------

    def get_account_no(self):
        return self.__account_no

    def get_name(self):
        return self.__name

    def get_balance_value(self):
        return self.__balance

    # --------------------------------------------------------
    # Setter for balance
    # --------------------------------------------------------

    def set_balance(self, balance):
        if balance < 0:
            raise InvalidAmountError("Balance cannot be negative.")

        self.__balance = balance

    # --------------------------------------------------------
    # Concrete method
    # --------------------------------------------------------

    def show_account_info(self):
        print("\n---------- ACCOUNT INFORMATION ----------")
        print("Account Number :", self.__account_no)
        print("Name           :", self.__name)
        print("Account Type   :", self.__class__.__name__)
        print("Balance        : ₹", self.__balance)

    # --------------------------------------------------------
    # Class method
    # --------------------------------------------------------

    @classmethod
    def show_total_accounts(cls):
        print("\nTotal Accounts Created:", Account.total_accounts)

    # --------------------------------------------------------
    # Generate account number
    # --------------------------------------------------------

    @classmethod
    def generate_account_no(cls):
        account_no = Account.next_account_no
        Account.next_account_no += 1
        return account_no


# ============================================================
# 3A. SAVINGS ACCOUNT
# ============================================================

class SavingsAccount(Account):

    DAILY_LIMIT = 25000
    MINIMUM_BALANCE = 1000
    INTEREST_RATE = 4

    def __init__(self, name, balance):
        account_no = Account.generate_account_no()

        # Using super()
        super().__init__(account_no, name, balance)

        self.__daily_withdrawn = 0
        self.__last_withdrawal_date = date.today()

    # --------------------------------------------------------
    # Check and reset daily withdrawal amount
    # --------------------------------------------------------

    def reset_daily_withdrawal(self):

        today = date.today()

        if today != self.__last_withdrawal_date:
            self.__daily_withdrawn = 0
            self.__last_withdrawal_date = today

    # --------------------------------------------------------
    # Deposit
    # --------------------------------------------------------

    def deposit(self, amount):

        if amount <= 0:
            raise InvalidAmountError(
                "Deposit amount must be greater than zero."
            )

        new_balance = self.get_balance_value() + amount
        self.set_balance(new_balance)

        print("₹", amount, "deposited successfully.")

    # --------------------------------------------------------
    # Withdraw
    # --------------------------------------------------------

    def withdraw(self, amount):

        self.reset_daily_withdrawal()

        if amount <= 0:
            raise InvalidAmountError(
                "Withdrawal amount must be greater than zero."
            )

        # Check daily limit
        if self.__daily_withdrawn + amount > self.DAILY_LIMIT:
            raise DailyLimitExceededError(
                "Daily withdrawal limit of ₹25,000 exceeded."
            )

        # Check minimum balance
        if self.get_balance_value() - amount < self.MINIMUM_BALANCE:
            raise InsufficientBalanceError(
                "Withdrawal would reduce balance below minimum "
                "balance of ₹1,000."
            )

        # Withdraw
        self.set_balance(self.get_balance_value() - amount)

        self.__daily_withdrawn += amount

        print("₹", amount, "withdrawn successfully.")
        print("Daily withdrawn: ₹", self.__daily_withdrawn)

    # --------------------------------------------------------
    # Get balance
    # --------------------------------------------------------

    def get_balance(self):
        return self.get_balance_value()

    # --------------------------------------------------------
    # Apply interest
    # --------------------------------------------------------

    def apply_interest(self):

        interest = self.get_balance_value() * self.INTEREST_RATE / 100

        self.set_balance(self.get_balance_value() + interest)

        print("Interest added: ₹", interest)

    # --------------------------------------------------------
    # Show information
    # --------------------------------------------------------

    def show_account_info(self):

        super().show_account_info()

        print("Interest Rate   :", self.INTEREST_RATE, "%")
        print("Minimum Balance :", "₹", self.MINIMUM_BALANCE)
        print("Daily Limit     :", "₹", self.DAILY_LIMIT)
        print("Daily Withdrawn :", "₹", self.__daily_withdrawn)


# ============================================================
# 3B. CURRENT ACCOUNT
# ============================================================

class CurrentAccount(Account):

    OVERDRAFT_LIMIT = 10000

    def __init__(self, name, balance):

        account_no = Account.generate_account_no()

        # Using super()
        super().__init__(account_no, name, balance)

    # --------------------------------------------------------
    # Deposit
    # --------------------------------------------------------

    def deposit(self, amount):

        if amount <= 0:
            raise InvalidAmountError(
                "Deposit amount must be greater than zero."
            )

        new_balance = self.get_balance_value() + amount
        self.set_balance(new_balance)

        print("₹", amount, "deposited successfully.")

    # --------------------------------------------------------
    # Withdraw
    # --------------------------------------------------------

    def withdraw(self, amount):

        if amount <= 0:
            raise InvalidAmountError(
                "Withdrawal amount must be greater than zero."
            )

        # Maximum amount that can be withdrawn
        # = current balance + overdraft limit

        maximum_withdrawal = (
            self.get_balance_value() + self.OVERDRAFT_LIMIT
        )

        if amount > maximum_withdrawal:
            raise InsufficientBalanceError(
                "Overdraft limit of ₹10,000 exceeded."
            )

        self.set_balance(self.get_balance_value() - amount)

        print("₹", amount, "withdrawn successfully.")

        if self.get_balance_value() < 0:
            print(
                "Overdraft used: ₹",
                abs(self.get_balance_value())
            )

    # --------------------------------------------------------
    # Get balance
    # --------------------------------------------------------

    def get_balance(self):
        return self.get_balance_value()

    # --------------------------------------------------------
    # Show information
    # --------------------------------------------------------

    def show_account_info(self):

        super().show_account_info()

        print("Overdraft Limit :", "₹", self.OVERDRAFT_LIMIT)
        print("Interest        : No Interest")


# ============================================================
# 4. BANKING SYSTEM
# ============================================================

class BankingSystem:

    def __init__(self):
        self.accounts = {}

    # --------------------------------------------------------
    # Create Savings Account
    # --------------------------------------------------------

    def create_savings_account(self, name, balance):

        if balance < SavingsAccount.MINIMUM_BALANCE:
            raise InvalidAmountError(
                "Savings account requires minimum initial "
                "balance of ₹1,000."
            )

        account = SavingsAccount(name, balance)

        self.accounts[account.get_account_no()] = account

        print("\nSavings Account created successfully.")
        print("Account Number:", account.get_account_no())

    # --------------------------------------------------------
    # Create Current Account
    # --------------------------------------------------------

    def create_current_account(self, name, balance):

        if balance < 0:
            raise InvalidAmountError(
                "Initial balance cannot be negative."
            )

        account = CurrentAccount(name, balance)

        self.accounts[account.get_account_no()] = account

        print("\nCurrent Account created successfully.")
        print("Account Number:", account.get_account_no())

    # --------------------------------------------------------
    # Find account
    # --------------------------------------------------------

    def find_account(self, account_no):

        if account_no not in self.accounts:
            raise AccountNotFoundError(
                "Account number does not exist."
            )

        return self.accounts[account_no]

    # --------------------------------------------------------
    # Deposit
    # --------------------------------------------------------

    def deposit_money(self, account_no, amount):

        account = self.find_account(account_no)

        account.deposit(amount)

    # --------------------------------------------------------
    # Withdraw
    # --------------------------------------------------------

    def withdraw_money(self, account_no, amount):

        account = self.find_account(account_no)

        account.withdraw(amount)

    # --------------------------------------------------------
    # Check balance
    # --------------------------------------------------------

    def check_balance(self, account_no):

        account = self.find_account(account_no)

        print("\nAccount Number:", account.get_account_no())
        print("Name:", account.get_name())
        print("Balance: ₹", account.get_balance())

    # --------------------------------------------------------
    # Show account information
    # --------------------------------------------------------

    def show_account_info(self, account_no):

        account = self.find_account(account_no)

        account.show_account_info()


# ============================================================
# 5. MENU
# ============================================================

def menu():

    bank = BankingSystem()

    while True:

        print("\n")
        print("==========================================")
        print("         BANKING MANAGEMENT SYSTEM")
        print("==========================================")
        print("1. Create Savings Account")
        print("2. Create Current Account")
        print("3. Deposit")
        print("4. Withdraw")
        print("5. Check Balance")
        print("6. Show Account Info")
        print("7. Show Total Accounts")
        print("8. Exit")
        print("==========================================")

        try:

            choice = int(input("Enter your choice: "))

            # ------------------------------------------------
            # CREATE SAVINGS ACCOUNT
            # ------------------------------------------------

            if choice == 1:

                name = input("Enter account holder name: ")

                balance = float(
                    input("Enter initial balance: ")
                )

                bank.create_savings_account(
                    name,
                    balance
                )

            # ------------------------------------------------
            # CREATE CURRENT ACCOUNT
            # ------------------------------------------------

            elif choice == 2:

                name = input("Enter account holder name: ")

                balance = float(
                    input("Enter initial balance: ")
                )

                bank.create_current_account(
                    name,
                    balance
                )

            # ------------------------------------------------
            # DEPOSIT
            # ------------------------------------------------

            elif choice == 3:

                account_no = int(
                    input("Enter account number: ")
                )

                amount = float(
                    input("Enter deposit amount: ")
                )

                bank.deposit_money(
                    account_no,
                    amount
                )

            # ------------------------------------------------
            # WITHDRAW
            # ------------------------------------------------

            elif choice == 4:

                account_no = int(
                    input("Enter account number: ")
                )

                amount = float(
                    input("Enter withdrawal amount: ")
                )

                bank.withdraw_money(
                    account_no,
                    amount
                )

            # ------------------------------------------------
            # CHECK BALANCE
            # ------------------------------------------------

            elif choice == 5:

                account_no = int(
                    input("Enter account number: ")
                )

                bank.check_balance(account_no)

            # ------------------------------------------------
            # ACCOUNT INFORMATION
            # ------------------------------------------------

            elif choice == 6:

                account_no = int(
                    input("Enter account number: ")
                )

                bank.show_account_info(account_no)

            # ------------------------------------------------
            # TOTAL ACCOUNTS
            # ------------------------------------------------

            elif choice == 7:

                Account.show_total_accounts()

            # ------------------------------------------------
            # EXIT
            # ------------------------------------------------

            elif choice == 8:

                print("Thank you for using the Banking System.")
                break

            # ------------------------------------------------
            # INVALID MENU OPTION
            # ------------------------------------------------

            else:

                print("Invalid choice. Please select 1-8.")

        # ====================================================
        # EXCEPTION HANDLING
        # ====================================================

        except ValueError:
            print(
                "Invalid input. Please enter a valid number."
            )

        except InvalidAmountError as e:
            print("Invalid Amount Error:", e)

        except InsufficientBalanceError as e:
            print("Insufficient Balance Error:", e)

        except AccountNotFoundError as e:
            print("Account Not Found Error:", e)

        except DailyLimitExceededError as e:
            print("Daily Limit Error:", e)

        except Exception as e:
            print("Unexpected Error:", e)

        # ====================================================
        # ELSE
        # ====================================================

        else:
            print("Transaction completed successfully.")

        # ====================================================
        # FINALLY
        # ====================================================

        finally:
            print("------------------------------------------")


# ============================================================
# 6. PROGRAM START
# ============================================================

if __name__ == "__main__":
    menu()



         BANKING MANAGEMENT SYSTEM
1. Create Savings Account
2. Create Current Account
3. Deposit
4. Withdraw
5. Check Balance
6. Show Account Info
7. Show Total Accounts
8. Exit


Enter your choice:  1
Enter account holder name:  Bhavani Karthikeyan
Enter initial balance:  10000



Savings Account created successfully.
Account Number: 1001
Transaction completed successfully.
------------------------------------------


         BANKING MANAGEMENT SYSTEM
1. Create Savings Account
2. Create Current Account
3. Deposit
4. Withdraw
5. Check Balance
6. Show Account Info
7. Show Total Accounts
8. Exit


Enter your choice:  2
Enter account holder name:  Ria
Enter initial balance:  4000



Current Account created successfully.
Account Number: 1002
Transaction completed successfully.
------------------------------------------


         BANKING MANAGEMENT SYSTEM
1. Create Savings Account
2. Create Current Account
3. Deposit
4. Withdraw
5. Check Balance
6. Show Account Info
7. Show Total Accounts
8. Exit


...

Invalid choice. Please select 1-8.
Transaction completed successfully.
------------------------------------------


         BANKING MANAGEMENT SYSTEM
1. Create Savings Account
2. Create Current Account
3. Deposit
4. Withdraw
5. Check Balance
6. Show Account Info
7. Show Total Accounts
8. Exit


Enter your choice:  5
Enter account number:  1001



Account Number: 1001
Name: Bhavani Karthikeyan
Balance: ₹ 1000.0
Transaction completed successfully.
------------------------------------------


         BANKING MANAGEMENT SYSTEM
1. Create Savings Account
2. Create Current Account
3. Deposit
4. Withdraw
5. Check Balance
6. Show Account Info
7. Show Total Accounts
8. Exit


Enter your choice:  8


Thank you for using the Banking System.
------------------------------------------
